# DRAMBender tutorial — writing DRAM Bender test programs

This notebook walks you from running the shipped templates to authoring your own JIT-compiled DRAM test programs on a DRAM Bender FPGA.

**Assumed environment**

- `bash build.sh` has succeeded and the `.venv/bin/python` kernel is active.
- A DDR4 FPGA is attached at `/dev/xdma0_*`.

**Assumed knowledge**

- DRAM fundamentals: bank/row/column, tRCD/tRAS/tRP/tWR, refresh, rowhammer.
- Basic Python and NumPy.

What this notebook teaches is the Python API — not DRAM itself.

**Sections**

0. DRAM Bender architecture at a glance
1. Setup
2. Hello DRAM — run the shipped templates
3. Inspect what you built
4. Single-sided rowhammer with shipped templates
5. Write your own program — `ProgramBuilder` primitives
6. Speed it up — `@program_template`
7. Capstone — fused rowhammer kernel
8. Where to go next

## 0. DRAM Bender architecture at a glance

DRAM Bender is a simple **in-order core** on the FPGA. It fetches one instruction per fabric cycle and executes it immediately — no out-of-order, no speculation beyond the taken-branch pipeline flush. Two kinds of instructions exist:

| kind | examples | time |
|---|---|---|
| **Bender instruction** (scalar / control) | `LI`, `ADDI`, `SUB`, `AND`, `LD`, `ST`, `LDWD`, `SLEEP`, `LABEL`, `BL`, `BEQ`, `JMP` | **6 ns per instruction** — one fabric cycle, except for branching and jumping which takes 6*6 = 36ns |
| **DRAM instruction** (mini-op, issued to the DRAM bus) | `PRE`, `ACT`, `RD`, `WR`, `REF`, `SEL_CH`, `NOP` | **1.5 ns per mini-op** — one mini-slot |

**One fabric cycle = 4 DRAM mini-slots.** When the core issues DRAM commands, it packs up to four mini-ops into one fabric word:

```
|  slot 0  |  slot 1  |  slot 2  |  slot 3  |    ← one fabric cycle (6 ns)
|  PRE     |  NOP     |  NOP     |  NOP     |    ← one p.DRAM(...) call
```

### Two DSL primitives for DRAM commands

- `p.DRAM(s0, s1, s2, s3)` — explicit 4-slot packing. Use `NOP()` for slots you don't fill. One fabric cycle.
- `p.DRAMSEQ(cmd(delay=N), ..., ALIGN())` — a sequence of DRAM commands separated by per-command slot delays, with `ALIGN()` as an optional terminator that pads to the next 4-slot boundary. Use for timed **multi-command** sequences. For a single command, use `p.DRAM(cmd, NOP, NOP, NOP)` + `p.SLEEP(N)` instead — single-op `DRAMSEQ` hides the slot layout.

### Other timing invariants

- **Taken branch = 6 fabric cycles** (pipeline flush). This dominates tight loops.
- **SLEEP(N)** expands to `DRAM(NOP, NOP, NOP, NOP)` cycles when `N ≤ 2`; it becomes one `SMC_SLEEP(N)` scalar instruction when `N ≥ 3`.

### DRAM timings to respect

Your programs must honor the DRAM's JEDEC timings — the FPGA doesn't check them. The three you'll see most:

| parameter | meaning | DDR4 minimum | slots | fabric cycles |
|---|---|---|---|---|
| **tRCD** | ACT → first RD/WR (same bank) | 18 ns | 12 | 3 |
| **tRAS** | ACT → PRE (same bank) | 36 ns | 24 | 6 |
| **tRP**  | PRE → ACT (same bank) | 18 ns | 12 | 3 |

`trace.summarize_timings()` reports observed min/max across a DRAM command trace — we'll use it in §3.

## 1. Setup

Opening the FPGA grabs `/dev/xdma0_*` until the board object is destroyed. In a notebook kernel that means **until the kernel dies** — so long-lived `board = open_board(...)` in cell output blocks every other process (and every other cell re-run). Every hardware cell below wraps its body in `with open_board(BoardType.DDR4, instance_id=0, host_interface=HostInterface.XDMA) as board: ...` — the device is released the moment the `with` block exits. You can re-run any cell without restarting the kernel.

The user-facing API lives under `drambender.api`. The DRAM command factories (ACT, NOP, PRE, RD, WR, REF, SEL_CH, ALIGN) have their own star-importable module at `drambender.api.program.instructions`.

In [ ]:
import sys
assert "/.venv" in sys.prefix, (
    f"Unexpected sys.prefix: {sys.prefix}. Select the .venv Python kernel."
)
print(sys.executable)

In [ ]:
import numpy as np

import drambender
from drambender.api import (
    BoardType,
    FinalProgram,
    HostInterface,
    ProgramBuilder,
    open_board,
    program_template,
)
from drambender.api.program.instructions import *

DDR4 geometry. These are **hardware facts** about the module under test, not defaults — the tool refuses to guess, because the wrong geometry silently produces wrong results on a different module.

In [ ]:
CACHELINES_PER_ROW  = 128
WORDS_PER_CACHELINE = 16
COLUMN_STRIDE       = 8
ROW_BYTES           = CACHELINES_PER_ROW * WORDS_PER_CACHELINE * 4  # 8192 on DDR4
ROW_BYTES

## 2. Hello DRAM — run the shipped templates

`drambender.builtin_programs.configure(...)` binds the shipped templates (`write_row`, `read_row`, `single_sided_rowhammer`, `double_sided_rowhammer`) to a concrete DRAM geometry.

All three kwargs are required — there is no default. See [CLAUDE.md §2(a)](../CLAUDE.md).

In [ ]:
bp = drambender.builtin_programs.configure(
    cachelines_per_row=CACHELINES_PER_ROW,
    column_stride=COLUMN_STRIDE,
    words_per_cacheline=WORDS_PER_CACHELINE,
)
bp

Write the pattern `0xDEADBEEF` to every word of one row; read the row back; compare.

Building a program (`bp.write_row(...)`, `bp.read_row(...)`) does **not** touch hardware — only `board.execute(...)` does. So we build the programs at module level (they're reused in §2 for inspection), then enter the `with` block only long enough to run them.

In [ ]:
BANK = 0
ROW = 0
PATTERN = 0xDEADBEEF

pattern_words = (PATTERN,) * WORDS_PER_CACHELINE
write_program = bp.write_row(BANK, ROW, pattern_words)
read_program  = bp.read_row(BANK, ROW)

readback = np.empty(CACHELINES_PER_ROW * WORDS_PER_CACHELINE, dtype=np.uint32)
with open_board(BoardType.DDR4, instance_id=0, host_interface=HostInterface.XDMA) as board:
    board.reset_fpga()
    board.execute([write_program, read_program])
    board.receive_into(readback)
    board.synchronize()

expected = np.full_like(readback, PATTERN)
assert np.array_equal(readback, expected), (
    f"{np.count_nonzero(readback != expected)} mismatches"
)
print(f"PASS: {readback.size} words matched (pattern=0x{PATTERN:08x})")

`board.execute([...])` submits a sequence of programs; `receive_into(buf)` pulls the next read's data into a pre-allocated NumPy buffer; `synchronize()` blocks until the FPGA has drained. The board is released the instant the `with` block exits — the follow-on verification runs with the FPGA already free.

## 3. Inspect what you built

Two tools you will use constantly — both CPU-only, no FPGA needed:

- `str(program)` — the decoded instruction listing (the C++ `debug::format_program` pretty-printer is bound to `__str__`).
- `program.trace_dram_commands()` — cycle-accurate DRAM command trace from the VM.

We inspect `write_program` from §1 — a `FinalProgram` sitting in memory, no board required.

In [ ]:
listing = str(write_program)
print(listing[:2000])  # first 2 KB of the listing

Each line is one 64-bit fabric instruction. DRAM lines show four mini-slots (16 bits each) encoded into a single cycle. Scalar ops (LI, ADDI, SLEEP, BL) take a full fabric cycle each.

Now the cycle-accurate DRAM command trace via the VM:

In [ ]:
trace = write_program.trace_dram_commands()
print(trace)

The delta (`Δt_next`) column is the wait to the next DRAM command; the absolute `t=` column is the cycle-accurate time from program start. See §0 for the slot-vs-cycle model.

`.summarize_timings()` walks the trace and reports min/max of tRCD, tRAS, and tRP — a quick way to check that a new program respects DDR4 spec, or to see by how much a deliberately-tight program (rowhammer!) violates it.

In [ ]:
print(write_program.trace_dram_commands().summarize_timings())

## 4. Single-sided rowhammer with shipped templates

`bp.single_sided_rowhammer(bank, aggressor_row, hammer_count)` repeatedly ACT/PRE's one row. You wrap it with writes (to initialise victim + aggressor patterns) and a read (to observe the victim).

The `Row` helper applies a **row mapping** — vendor-specific physical↔logical swizzling — and carries the write pattern through any bitline/DQ swizzle the module exposes. `"MI1"` is one such mapping shipped with the tool; see [python/drambender/rows/mappings/](../python/drambender/rows/mappings/) for the others.

The sweep below runs 5 victim rows under one `with open_board(...)` — opening the board per iteration would be correct but wasteful. Open once, run the whole sweep, release at the `with` boundary.

In [ ]:
HAMMER_COUNT = 500_000
NUM_VICTIMS  = 5
START_ROW    = 81
ROW_WORDS    = ROW_BYTES // 4

total_bitflips = 0

with open_board(BoardType.DDR4, instance_id=0, host_interface=HostInterface.XDMA) as board:
    board.reset_fpga()

    for i in range(NUM_VICTIMS):
        victim_physical    = START_ROW + i
        aggressor_physical = victim_physical + 1

        victim = drambender.rows.Row(
            physical_id=victim_physical,
            row_mapping="Linear",
            data_pattern=0x00000000,
        )
        aggressor = drambender.rows.Row(
            physical_id=aggressor_physical,
            row_mapping="Linear",
            data_pattern=0xFFFFFFFF,
        )

        board.execute([
            bp.write_row(BANK, victim.logical_id,    victim.write_pattern),
            bp.write_row(BANK, aggressor.logical_id, aggressor.write_pattern),
            bp.single_sided_rowhammer(BANK, aggressor.logical_id, HAMMER_COUNT),
            bp.read_row(BANK, victim.logical_id),
        ])
        rb = np.empty(ROW_WORDS, dtype=np.uint32)
        board.receive_into(rb)
        board.synchronize()
        pattern  = np.asarray(victim.write_pattern, dtype=np.uint32)
        expected = np.tile(pattern, ROW_WORDS // len(pattern))
        mask     = rb ^ expected

        n_flips = int(np.unpackbits(mask.view(np.uint8), bitorder="little").sum())
        total_bitflips += n_flips
        print(
            f"row {victim_physical:5d} -> {victim.logical_id:5d} "
            f"(hammer {aggressor_physical}): {n_flips} flips"
        )

print(f"\ntotal: {total_bitflips} bitflips across {NUM_VICTIMS} victims")

Readback is explicit: allocate a NumPy buffer with the right dtype/shape, call `board.receive_into(buf)`, then `board.synchronize()`. You own the XOR step, which keeps the comparison close to whatever bitline / DQ pattern mapping you're layering on top. `np.tile` broadcasts a short cacheline-sized pattern across the whole row (`ROW_WORDS // len(pattern)` copies). DDR4 rows are 8192 bytes and HBM2 rows are 4096 — always size your buffer to the module under test.

## 5. Write your own program — `ProgramBuilder` primitives

`ProgramBuilder` is a Python DSL that emits DRAM Bender fabric instructions. The workflow:

1. Allocate named registers (or use the reserved ones below).
2. Load values with `LI` (load immediate) / `ADDI` (add immediate).
3. Broadcast scalar patterns into the wide register with `LDWD`.
4. Issue DRAM commands via `DRAM(...)` (4 mini-slots per cycle) or `DRAMSEQ(..., ALIGN())` for multi-command sequences.
5. Sleep or loop with `SLEEP(N)`, `LABEL`, `BL`.
6. Finalize with `p.conclude()` — returns a `FinalProgram` ready for `board.execute(...)`.

### Register allocation

DRAM Bender has **16 total registers**. The first 7 are pre-allocated to well-known names and used by the DRAM-command operands (bank, row, column, stride, wide pattern). The remaining 9 are yours to allocate by name.

| id | name | typical use |
|----|------|-------------|
| 0 | `CASR` | column-address step register |
| 1 | `BASR` | bank-address step register |
| 2 | `RASR` | row-address step register |
| 3 | `CAR`  | current column |
| 4 | `BAR`  | current bank |
| 5 | `RAR`  | current row |
| 6 | `PATTERN_REG` | wide write pattern |
| 7–15 | user-defined | `p.alloc_reg("MY_NAME")` |

`p.alloc_reg(name)` returns the register id and memoizes the mapping. Passing a register name (string) to any builder method (`LI`, `ADDI`, `LD`, `ST`, `BL`, ...) transparently allocates it on first use and reuses it afterward. There is no register spilling — allocating a 10th user register raises `ValueError`. When that happens, stage intermediate values in SoftMC scratch memory via `LD`/`ST` instead.

```python
p = ProgramBuilder()
p.alloc_reg("NUM_HMR")       # explicitly allocated — id 7
p.LI(1000, "NUM_HMR")        # fine
p.LI(0, "HMR_COUNTER")       # implicit allocation on first use — id 8
p.ADDI("HMR_COUNTER", 1, "HMR_COUNTER")
```

### The rule to memorize

**Do not use** `p.PRE()`, `p.ACT()`, `p.RD()`, `p.WR()`, `p.REF()`, `p.SEL_CH()` as top-level builder calls — they are stubs that raise `NotImplementedError` on purpose. They would hide which mini-slot your command lands in, and packing determines your hammer rate.

**Do use**:

- `p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())` — explicit 4-slot packing.
- `p.DRAMSEQ(PRE("BAR", delay=12))` — one command + 11 NOP cycles.

Below, we write one cacheline (not the whole row) from scratch, then verify it.

In [ ]:
def build_write_cacheline(
    bank: int, row: int, col: int, pattern: int,
) -> FinalProgram:
    p = ProgramBuilder()
    p.LI(bank, "BAR")
    p.LI(row,  "RAR")
    p.LI(col,  "CAR")
    p.LI(COLUMN_STRIDE, "CASR")

    # Broadcast scalar pattern into every lane of PATTERN_REG.
    for lane in range(WORDS_PER_CACHELINE):
        p.LI(pattern, "PATTERN_REG")
        p.LDWD("PATTERN_REG", lane)

    # PRE (precharge any open row), then ACT, one WR, close.
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    p.DRAM(ACT("BAR", "RAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    p.DRAM(WR("BAR", "CAR"), NOP(), NOP(), NOP())
    p.SLEEP(1)
    p.SLEEP(8)
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(3)  # tRP cushion
    return p.conclude()

Zero the row first, then write one cacheline, then read the row and verify only that cacheline changed.

In [ ]:
CHECK_ROW     = 5
CHECK_COL     = 0
CHECK_PATTERN = 0xCAFEBABE

readback = np.empty(CACHELINES_PER_ROW * WORDS_PER_CACHELINE, dtype=np.uint32)
with open_board(BoardType.DDR4, instance_id=0, host_interface=HostInterface.XDMA) as board:
    board.execute([
        bp.write_row(BANK, CHECK_ROW, (0,) * WORDS_PER_CACHELINE),
        build_write_cacheline(BANK, CHECK_ROW, CHECK_COL, CHECK_PATTERN),
        bp.read_row(BANK, CHECK_ROW),
    ])
    board.receive_into(readback)
    board.synchronize()

first_cacheline = readback[:WORDS_PER_CACHELINE]
rest            = readback[WORDS_PER_CACHELINE:]

assert np.all(first_cacheline == CHECK_PATTERN), "first cacheline should hold the pattern"
assert np.all(rest == 0),                         "rest of row should be zero"
print(
    f"PASS: cacheline {CHECK_COL} = 0x{CHECK_PATTERN:08x}; "
    f"other {CACHELINES_PER_ROW - 1} cachelines are 0"
)

Diff your cacheline write against the full-row template in [examples/read_write_sanity_jit.py](../examples/read_write_sanity_jit.py): the shipped version loops `WR("BAR", "CAR", icar=1, delay=8)` 128 times and relies on `icar=1` to auto-increment `CAR` by `CASR` each cycle.

## 6. Speed it up — `@program_template`

In Section 3 we rebuilt the program in Python on every loop iteration. For large sweeps the Python build cost dominates. `@program_template` fixes this:

1. **First call** — trace the builder, emit a C++ plugin, compile, load, cache. Cost: a few hundred ms (cold) or ~5 ms (disk cache hit, `compiled_warm_disk`).
2. **Subsequent calls with the same scalar shape** — patch args into the loaded plugin. Cost: microseconds (`compiled_hot`).

Loops with constant trip counts are unrolled at trace time, so `for _ in range(128)` in your build function becomes 128 unrolled mini-ops in the compiled plugin.

If the JIT can't find a compiler it falls back to interpreted mode with a one-time `RuntimeWarning` per template — the template still runs, just not accelerated.

Note: JIT stats below are pure-CPU — building a program doesn't touch hardware, only executing it does.

In [ ]:
@program_template
def build_write_cacheline_jit(
    bank: int, row: int, col: int, pattern: int,
) -> FinalProgram:
    p = ProgramBuilder()
    p.LI(bank, "BAR")
    p.LI(row,  "RAR")
    p.LI(col,  "CAR")
    p.LI(COLUMN_STRIDE, "CASR")
    for lane in range(WORDS_PER_CACHELINE):
        p.LI(pattern, "PATTERN_REG")
        p.LDWD("PATTERN_REG", lane)
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    p.DRAM(ACT("BAR", "RAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    p.DRAM(WR("BAR", "CAR"), NOP(), NOP(), NOP())
    p.SLEEP(1)
    p.SLEEP(8)
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(3)  # tRP cushion
    return p.conclude()

In [ ]:
from drambender.api.jit import get_last_template_run_stats

# First call: cold (compiled_cold if no on-disk .so, compiled_warm_disk if cache hit).
_ = build_write_cacheline_jit(BANK, 0, 0, 0xDEADBEEF)
s1 = get_last_template_run_stats()
print(
    f"call 1: mode={s1.mode}, cache_hit={s1.cache_hit}, "
    f"trace_s={s1.trace_s:.4f}, compile_s={s1.compile_s:.4f}"
)

# Five subsequent calls with the same scalar shape → hot.
for _ in range(5):
    _ = build_write_cacheline_jit(BANK, 0, 0, 0xDEADBEEF)
s2 = get_last_template_run_stats()
print(f"call N: mode={s2.mode}")

To inspect or wipe the JIT cache on disk use `drambender.api.jit.{set_jit_cache_dir, clear_template_caches}`. By default the cache lives at `build/jit-cache/` — safe to `rm -rf` at any time; the next run will recompile.

## 7. Capstone — fused rowhammer kernel

Compose everything into ONE JIT-compiled template that does victim-init → aggressor-init → hammer loop → victim readback in a single submission. This mirrors [examples/single_sided_rowhammer_jit.py](../examples/single_sided_rowhammer_jit.py) — use that as the reference implementation.

The hammer loop packs:

```
p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())   # cycle 1: PRE in slot 0
p.ADDI("HMR_COUNTER", 1, "HMR_COUNTER")   # cycle 2: scalar op
p.DRAM(NOP(), NOP(), NOP(), ACT("BAR", "RAR"))  # cycle 3: ACT in slot 3
p.BL("HMR_COUNTER", "NUM_HMR", "HMR_BEGIN")  # cycle 4 + 5 cycles pipeline flush on branch taken
```

Read the VM trace of this program if you want to be sure about the per-iteration rate.

In [ ]:
@program_template
def build_rowhammer_program(
    bank: int,
    victim_row: int,
    aggressor_row: int,
    victim_pattern: int,
    aggressor_pattern: int,
    hammer_count: int,
):
    p = ProgramBuilder()
    p.alloc_reg("NUM_HMR")
    p.alloc_reg("HMR_COUNTER")

    p.LI(bank, "BAR")
    p.LI(COLUMN_STRIDE, "CASR")

    # --- Initialise victim ---
    p.LI(victim_row, "RAR")
    for lane in range(WORDS_PER_CACHELINE):
        p.LI(victim_pattern, "PATTERN_REG")
        p.LDWD("PATTERN_REG", lane)
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.LI(0, "CAR")
    p.SLEEP(2)
    p.DRAM(ACT("BAR", "RAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    for _ in range(CACHELINES_PER_ROW):
        p.DRAM(WR("BAR", "CAR", icar=1), NOP(), NOP(), NOP())
        p.SLEEP(1)
    p.SLEEP(8)
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    # --- Initialise aggressor ---
    p.LI(aggressor_row, "RAR")
    for lane in range(WORDS_PER_CACHELINE):
        p.LI(aggressor_pattern, "PATTERN_REG")
        p.LDWD("PATTERN_REG", lane)
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.LI(0, "CAR")
    p.SLEEP(2)
    p.DRAM(ACT("BAR", "RAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    for _ in range(CACHELINES_PER_ROW):
        p.DRAM(WR("BAR", "CAR", icar=1), NOP(), NOP(), NOP())
        p.SLEEP(1)
    p.SLEEP(8)
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    # --- Hammer the aggressor ---
    p.LI(aggressor_row, "RAR")
    p.LI(0, "HMR_COUNTER")
    p.LI(hammer_count, "NUM_HMR")
    p.LABEL("HMR_BEGIN")
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.ADDI("HMR_COUNTER", 1, "HMR_COUNTER")
    p.DRAM(NOP(), NOP(), NOP(), ACT("BAR", "RAR"))
    p.BL("HMR_COUNTER", "NUM_HMR", "HMR_BEGIN")
    # tRAS for the last aggressor ACT — BL-not-taken (loop exit) is only
    # 1 cycle, versus 6 cycles on BL-taken.
    p.SLEEP(5)

    # --- Read victim back ---
    p.LI(victim_row, "RAR")
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.LI(0, "CAR")
    p.SLEEP(2)
    p.DRAM(ACT("BAR", "RAR"), NOP(), NOP(), NOP())
    p.SLEEP(2)
    for _ in range(CACHELINES_PER_ROW):
        p.DRAM(RD("BAR", "CAR", icar=1), NOP(), NOP(), NOP())
        p.SLEEP(1)
    p.SLEEP(4)
    p.DRAM(PRE("BAR"), NOP(), NOP(), NOP())
    p.SLEEP(3)  # tRP cushion
    return p.conclude()

In [ ]:
HAMMER_COUNT_SWEEP = 500_000
NUM_VICTIMS_SWEEP  = 30
FLIP_PREVIEW_LIMIT = 6

vulnerable = 0


def flip_locations(mask: np.ndarray, *, limit: int) -> list[tuple[int, int, int]]:
    """Decode flipped bit positions as (cacheline, word_in_cacheline, bit_in_word)."""
    bit_indices = np.flatnonzero(np.unpackbits(mask.view(np.uint8), bitorder="little"))
    locations = []
    for bit_index in bit_indices[:limit]:
        word_index        = int(bit_index) // 32
        cacheline         = word_index // WORDS_PER_CACHELINE
        word_in_cacheline = word_index % WORDS_PER_CACHELINE
        bit_in_word       = int(bit_index) % 32
        locations.append((cacheline, word_in_cacheline, bit_in_word))
    return locations


with open_board(BoardType.DDR4, instance_id=0, host_interface=HostInterface.XDMA) as board:
    board.reset_fpga()

    for i in range(NUM_VICTIMS_SWEEP):
        victim_physical    = START_ROW + i
        aggressor_physical = victim_physical + 1

        victim    = drambender.rows.Row(
            physical_id=victim_physical,    row_mapping="Linear", data_pattern=0x00000000,
        )
        aggressor = drambender.rows.Row(
            physical_id=aggressor_physical, row_mapping="Linear", data_pattern=0xFFFFFFFF,
        )

        program = build_rowhammer_program(
            bank=BANK,
            victim_row=victim.logical_id,
            aggressor_row=aggressor.logical_id,
            victim_pattern=0x00000000,
            aggressor_pattern=0xFFFFFFFF,
            hammer_count=HAMMER_COUNT_SWEEP,
        )

        board.execute(program)
        rb = np.empty(ROW_WORDS, dtype=np.uint32)
        board.receive_into(rb)
        board.synchronize()
        pattern  = np.asarray(victim.write_pattern, dtype=np.uint32)
        expected = np.tile(pattern, ROW_WORDS // len(pattern))
        mask     = rb ^ expected

        n_flips = int(np.unpackbits(mask.view(np.uint8), bitorder="little").sum())
        if n_flips:
            vulnerable += 1
            locations = flip_locations(mask, limit=FLIP_PREVIEW_LIMIT)
            preview = ", ".join(f"cl={cl} w={w} b={b}" for cl, w, b in locations)
            suffix = "" if n_flips <= FLIP_PREVIEW_LIMIT else f", +{n_flips - FLIP_PREVIEW_LIMIT} more"
            print(
                f"row {victim_physical:5d} -> {victim.logical_id:5d} "
                f"(aggressor {aggressor_physical}): {n_flips} flips @ {preview}{suffix}"
            )

print(f"\n{vulnerable}/{NUM_VICTIMS_SWEEP} rows vulnerable")

# Peek at the timing profile of one generated rowhammer program — this is
# the per-iteration cadence, not the whole sweep.
print(program.trace_dram_commands().summarize_timings())

## 8. Where to go next

- **Other shipped templates**: `drambender.builtin_programs.double_sided_rowhammer` — sources in [python/drambender/builtin_programs/](../python/drambender/builtin_programs/).
- **Row mappings for vendor swizzles**: `drambender.rows.mappings.{linear, sa0, mi1}`.
- **Bitline / DQ pattern mappings**: `drambender.patterns.{bitline_mappings, dq_mappings}`.
- **HBM2**: open with `BoardType.HBM2` instead of `DDR4`; channels select with `SEL_CH(...)`; row size is **4096 bytes** (not 8192) — size the `receive_into` buffer accordingly.
- **JIT debugging**: `drambender.api.jit.{clear_template_caches, set_jit_cache_dir, get_last_template_run_stats}`. On-disk cache at `build/jit-cache/`.
- **Raw SMC instruction factories** (for direct `Program.add_inst(...)` construction): `drambender.api.program.raw_instructions`.
- **Full API reference = the source**. Start at [python/drambender/api/program/builder.py](../python/drambender/api/program/builder.py) and [python/drambender/builtin_programs/](../python/drambender/builtin_programs/).

No explicit cleanup needed — every hardware cell released the board on `with` exit. You can re-run any section without restarting the kernel.